In [1]:
# ─────────────────────────────────────────
# CART — Classification and Regression Tree (Gini Impurity)
# ─────────────────────────────────────────
from collections import Counter

# Données : [Météo, Vent, Jouer?]
dataset = [
    ["Soleil","Faible","Oui"], ["Soleil","Fort","Non"],
    ["Nuage","Faible","Oui"], ["Pluie","Faible","Oui"],
    ["Pluie","Fort","Non"],   ["Nuage","Fort","Oui"],
    ["Soleil","Fort","Non"],  ["Pluie","Faible","Oui"],
]
features = ["Météo", "Vent"]

# Indice de Gini : mesure l'impureté d'un nœud
# Gini = 1 - Σ(p_i)²  → 0 = pur, 0.5 = max d'impureté
def gini(labels):
    n = len(labels)
    if n == 0: return 0
    return 1 - sum((c/n)**2 for c in Counter(labels).values())

# Gini pondéré d'un split sur la colonne col
def gini_split(data, col):
    n = len(data)
    vals = set(r[col] for r in data)
    return sum(
        (len(sub := [r for r in data if r[col] == v]) / n) * gini([r[-1] for r in sub])
        for v in vals
    )

# Construction récursive de l'arbre CART
def cart(data, feats, profondeur=0):
    labels = [r[-1] for r in data]
    if len(set(labels)) == 1: return labels[0]          # Feuille pure
    if not feats or profondeur > 3:                     # Limite de profondeur
        return Counter(labels).most_common(1)[0][0]
    # Choisir l'attribut avec le Gini le plus BAS (= plus pur)
    best = min(feats, key=lambda f: gini_split(data, features.index(f)))
    col  = features.index(best)
    tree = {best: {}}
    for val in set(r[col] for r in data):
        sous = [r for r in data if r[col] == val]
        tree[best][val] = cart(sous, [f for f in feats if f != best], profondeur+1)
    return tree

arbre = cart(dataset, features)
print("Arbre CART :", arbre)
# → {'Météo': {'Nuage': 'Oui', 'Pluie': 'Oui', 'Soleil': 'Non'}}


Arbre CART : {'Vent': {'Faible': 'Oui', 'Fort': {'Météo': {'Soleil': 'Non', 'Pluie': 'Non', 'Nuage': 'Oui'}}}}
